In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Create folder to save charts and outputs
os.makedirs("outputs", exist_ok=True)

# 1. Load the raw dataset
df = pd.read_csv("/content/customer_churn_raw.csv")

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())

print("\nData Types & Non-Null Counts:")
print(df.info())

print("\nMissing Values per Column:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nChurn Distribution:")
print(df["churn"].value_counts())

Dataset Shape: (1208, 12)

First 5 Rows:
  customer_id  gender  senior_citizen  tenure_months   contract_type  \
0   CUST-1001    Male               0              5  Month-to-Month   
1   CUST-1002  Female               0             20  Month-to-Month   
2   CUST-1003    Male               0             21  Month-to-Month   
3   CUST-1004    Male               0             58        Two Year   
4   CUST-1005    Male               0             31  Month-to-Month   

  internet_service tech_support    payment_method  support_tickets  \
0      Fiber Optic          Yes     UPI / AutoPay                3   
1      No Internet  No Internet     UPI / AutoPay                3   
2      No Internet  No Internet     Bank Transfer                2   
3      Fiber Optic          Yes  Electronic Check                3   
4      Fiber Optic           No       Credit Card                4   

   monthly_charges total_charges churn  
0            86.97        412.24   Yes  
1            20.86     

In [2]:
# --- STEP 2: DATA CLEANING & PREPROCESSING ---

# 1. Remove duplicate customer records
df = df.drop_duplicates(subset=["customer_id"]).copy()

# 2. Standardize inconsistent text casing in contract_type
df["contract_type"] = df["contract_type"].str.strip().str.title()

# 3. Convert total_charges from text to numeric (blank spaces become NaN)
df["total_charges"] = pd.to_numeric(df["total_charges"], errors="coerce")

# 4. Fill missing numeric and categorical values
df["monthly_charges"] = df["monthly_charges"].fillna(df["monthly_charges"].median())
df["total_charges"] = df["total_charges"].fillna(df["monthly_charges"] * df["tenure_months"])
df["internet_service"] = df["internet_service"].fillna(df["internet_service"].mode()[0])

# 5. Save the cleaned dataset
df.to_csv("customer_churn_cleaned.csv", index=False)

print("\n--- After Cleaning ---")
print("Cleaned Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())
print("total_charges Data Type:", df["total_charges"].dtype)
print("\nStandardized Contract Types:")
print(df["contract_type"].value_counts())


--- After Cleaning ---
Cleaned Shape: (1200, 12)
Total Missing Values: 0
total_charges Data Type: float64

Standardized Contract Types:
contract_type
Month-To-Month    657
One Year          292
Two Year          251
Name: count, dtype: int64


In [3]:
# --- STEP 3: EXPLORATORY DATA ANALYSIS (EDA) ---

sns.set_theme(style="whitegrid")

# 1. Overall churn rate
overall_churn = (df["churn"] == "Yes").mean() * 100
print(f"\nOverall Churn Rate: {overall_churn:.2f}%")

# 2. Churn percentage across contract types
contract_churn = pd.crosstab(df["contract_type"], df["churn"], normalize="index") * 100
print("\nChurn Rate (%) by Contract Type:")
print(contract_churn.round(2))

# 3. Compare average tenure, monthly charges, and support tickets
numeric_summary = df.groupby("churn")[["tenure_months", "monthly_charges", "support_tickets"]].mean().round(2)
print("\nAverage Metrics by Churn Status:")
print(numeric_summary)

# 4. Chart 1: Churn Count by Contract Type
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.countplot(
    data=df,
    x="contract_type",
    hue="churn",
    hue_order=["No", "Yes"],
    palette=["#2E86AB", "#E05A47"],
    ax=ax
)
ax.set_title("Customer Churn Count by Contract Type", fontsize=13, fontweight="bold")
ax.set_xlabel("Contract Type")
ax.set_ylabel("Number of Customers")
ax.legend(title="Churned")
plt.tight_layout()
plt.savefig("outputs/churn_by_contract.png", dpi=150)
plt.close()

# 5. Chart 2: Tenure & Support Tickets vs. Churn
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

sns.boxplot(
    data=df,
    x="churn",
    y="tenure_months",
    order=["No", "Yes"],
    hue="churn",
    hue_order=["No", "Yes"],
    palette=["#2E86AB", "#E05A47"],
    ax=axes[0]
)
if axes[0].get_legend():
    axes[0].get_legend().remove()
axes[0].set_title("Tenure (Months) vs. Churn", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Churned")
axes[0].set_ylabel("Tenure (Months)")

sns.barplot(
    data=df,
    x="churn",
    y="support_tickets",
    order=["No", "Yes"],
    hue="churn",
    hue_order=["No", "Yes"],
    palette=["#2E86AB", "#E05A47"],
    ax=axes[1]
)
if axes[1].get_legend():
    axes[1].get_legend().remove()
axes[1].set_title("Average Support Tickets vs. Churn", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Churned")
axes[1].set_ylabel("Avg Support Tickets")

plt.tight_layout()
plt.savefig("outputs/tenure_and_tickets_vs_churn.png", dpi=150)
plt.close()

print("\nSaved EDA charts to outputs/")


Overall Churn Rate: 32.75%

Churn Rate (%) by Contract Type:
churn              No    Yes
contract_type               
Month-To-Month  54.95  45.05
One Year        77.74  22.26
Two Year        87.25  12.75

Average Metrics by Churn Status:
       tenure_months  monthly_charges  support_tickets
churn                                                 
No             39.42            64.96             1.79
Yes            29.32            78.20             2.54

Saved EDA charts to outputs/


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- STEP 4: FEATURE ENGINEERING & DATA ENCODING ---

# 1. Create domain-specific behavioral features
df["avg_monthly_spend"] = np.round(df["total_charges"] / df["tenure_months"], 2)
df["tickets_per_month"] = np.round(df["support_tickets"] / df["tenure_months"], 3)
df["is_new_customer"] = np.where(df["tenure_months"] <= 6, 1, 0)

# 2. Encode target variable (Yes -> 1, No -> 0)
df["churn_flag"] = df["churn"].map({"Yes": 1, "No": 0})

# 3. Select features and apply One-Hot Encoding to categorical columns
feature_cols = [
    "gender", "senior_citizen", "tenure_months", "contract_type",
    "internet_service", "tech_support", "payment_method",
    "support_tickets", "monthly_charges", "total_charges",
    "avg_monthly_spend", "tickets_per_month", "is_new_customer"
]

X = pd.get_dummies(df[feature_cols], drop_first=True, dtype=int)
y = df["churn_flag"]

# 4. Split into 80% Training and 20% Testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Scale numeric features for Logistic Regression (fit only on training data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n--- Feature Engineering & Split Summary ---")
print("Total Encoded Features:", X.shape[1])
print("Training Set Shape:", X_train.shape)
print("Testing Set Shape:", X_test.shape)
print("Training Churn Rate:", round(y_train.mean() * 100, 2), "%")
print("Testing Churn Rate:", round(y_test.mean() * 100, 2), "%")


--- Feature Engineering & Split Summary ---
Total Encoded Features: 18
Training Set Shape: (960, 18)
Testing Set Shape: (240, 18)
Training Churn Rate: 32.71 %
Testing Churn Rate: 32.92 %


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# --- STEP 5: TRAIN & COMPARE MODELS ---

# 1. Train Logistic Regression (Baseline Model on scaled features)
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

# 2. Train Random Forest Classifier (Tree-Based Ensemble)
rf_model = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# 3. Compare Model Metrics
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [accuracy_score(y_test, lr_pred), accuracy_score(y_test, rf_pred)],
    "Precision": [precision_score(y_test, lr_pred), precision_score(y_test, rf_pred)],
    "Recall": [recall_score(y_test, lr_pred), recall_score(y_test, rf_pred)],
    "F1-Score": [f1_score(y_test, lr_pred), f1_score(y_test, rf_pred)],
    "ROC-AUC": [roc_auc_score(y_test, lr_prob), roc_auc_score(y_test, rf_prob)]
}).round(3)

print("\n--- Model Comparison Summary ---")
print(results.to_string(index=False))

print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test, rf_pred, target_names=["Retained (0)", "Churned (1)"]))

# 4. Chart 3: Side-by-Side Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, preds, title in zip(
    axes,
    [lr_pred, rf_pred],
    ["Logistic Regression Confusion Matrix", "Random Forest Confusion Matrix"]
):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False,
        xticklabels=["Retained", "Churned"],
        yticklabels=["Retained", "Churned"],
        ax=ax
    )
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("Actual Label")

plt.tight_layout()
plt.savefig("outputs/confusion_matrices.png", dpi=150)
plt.close()

# 5. Chart 4: Top 10 Feature Importances (Random Forest)
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, color="#2E86AB", ax=ax)
ax.set_title("Top 10 Drivers of Customer Churn (Random Forest)", fontsize=12, fontweight="bold")
ax.set_xlabel("Feature Importance Score")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.savefig("outputs/feature_importance.png", dpi=150)
plt.close()

print("\nTop 5 Churn Drivers:")
print(importances.head(5).round(3))
print("\nSaved confusion_matrices.png and feature_importance.png to outputs/")


--- Model Comparison Summary ---
              Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
Logistic Regression     0.742      0.660   0.443     0.530    0.801
      Random Forest     0.742      0.639   0.494     0.557    0.789

--- Random Forest Classification Report ---
              precision    recall  f1-score   support

Retained (0)       0.78      0.86      0.82       161
 Churned (1)       0.64      0.49      0.56        79

    accuracy                           0.74       240
   macro avg       0.71      0.68      0.69       240
weighted avg       0.73      0.74      0.73       240


Top 5 Churn Drivers:
tickets_per_month    0.170
avg_monthly_spend    0.149
monthly_charges      0.138
tenure_months        0.119
total_charges        0.115
dtype: float64

Saved confusion_matrices.png and feature_importance.png to outputs/


In [6]:
# --- STEP 6: RETENTION RISK SCORING & ACTIONABLE EXPORT ---

# 1. Build test-set risk scorecard using Random Forest probabilities
risk_df = df.loc[X_test.index, [
    "customer_id", "contract_type", "tenure_months",
    "monthly_charges", "support_tickets", "churn"
]].copy()

risk_df["churn_probability"] = np.round(rf_prob, 3)

# 2. Segment customers into Low, Medium, and High Churn Risk tiers
risk_df["risk_tier"] = pd.cut(
    risk_df["churn_probability"],
    bins=[-0.01, 0.30, 0.60, 1.00],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

# 3. Assign recommended retention action per tier
action_map = {
    "Low Risk": "Standard Loyalty Newsletter",
    "Medium Risk": "Proactive Check-In & Tech Support Offer",
    "High Risk": "Priority Retention Call + Annual Plan Discount"
}
risk_df["recommended_action"] = risk_df["risk_tier"].map(action_map)

# 4. Sort highest-risk customers first and export CSV
risk_df = risk_df.sort_values(by="churn_probability", ascending=False)
risk_df.to_csv("outputs/customer_retention_risk_scores.csv", index=False)

print("\n--- Retention Risk Tier Distribution (Test Set) ---")
tier_summary = risk_df.groupby("risk_tier", observed=False).agg(
    customer_count=("customer_id", "count"),
    avg_churn_prob=("churn_probability", "mean"),
    monthly_revenue_at_risk=("monthly_charges", "sum")
).round(2)
print(tier_summary)

print("\n--- Top 5 Highest-Risk Customers ---")
print(risk_df.head(5).to_string(index=False))

# 5. Chart 5: Customer Distribution by Retention Risk Tier
fig, ax = plt.subplots(figsize=(7, 4.5))
tier_order = ["Low Risk", "Medium Risk", "High Risk"]
sns.countplot(
    data=risk_df,
    x="risk_tier",
    order=tier_order,
    hue="risk_tier",
    hue_order=tier_order,
    palette=["#2E86AB", "#F4A261", "#E05A47"],
    ax=ax
)
if ax.get_legend():
    ax.get_legend().remove()
ax.set_title("Test Customers Segmented by Retention Risk Tier", fontsize=12, fontweight="bold")
ax.set_xlabel("Retention Risk Tier")
ax.set_ylabel("Number of Customers")
plt.tight_layout()
plt.savefig("outputs/retention_risk_tiers.png", dpi=150)
plt.close()

print("\nSaved customer_retention_risk_scores.csv and retention_risk_tiers.png to outputs/")


--- Retention Risk Tier Distribution (Test Set) ---
             customer_count  avg_churn_prob  monthly_revenue_at_risk
risk_tier                                                           
Low Risk                120            0.18                  7203.26
Medium Risk              80            0.44                  6426.25
High Risk                40            0.67                  3557.49

--- Top 5 Highest-Risk Customers ---
customer_id  contract_type  tenure_months  monthly_charges  support_tickets churn  churn_probability risk_tier                             recommended_action
  CUST-1495 Month-To-Month             11            99.60                3   Yes              0.821 High Risk Priority Retention Call + Annual Plan Discount
  CUST-1387       One Year             11           111.04                3   Yes              0.740 High Risk Priority Retention Call + Annual Plan Discount
  CUST-2111 Month-To-Month             27            98.22                3   Yes         